# Estimation Theory - Exercises

Practice problems for statistical estimation concepts.

## Exercises
1. Bias Calculation
2. MLE for Poisson Distribution
3. Fisher Information
4. Method of Moments for Beta Distribution
5. MSE Comparison

In [ ]:
import numpy as np
from scipy import stats
from scipy.optimize import minimize
import matplotlib.pyplot as plt

np.random.seed(42)

---
## Exercise 1: Bias Calculation

**Problem**: Investigate the bias of sample standard deviation.

Tasks:
1. Simulate 10,000 samples of size n=5 from N(0, σ²=4)
2. Calculate biased (ddof=0) and unbiased (ddof=1) std for each
3. Compare E[ŝ] to true σ=2 for both estimators
4. Which one has less bias? Why isn't ddof=1 perfectly unbiased for std?

In [ ]:
# Parameters
true_sigma = 2
n = 5
n_simulations = 10000

# Your solution here

### Solution

In [ ]:
# Parameters
true_sigma = 2
n = 5
n_simulations = 10000

print("BIAS OF STANDARD DEVIATION ESTIMATORS")
print("="*55)

np.random.seed(42)

# 1. Simulate samples
stds_biased = []
stds_unbiased = []

for _ in range(n_simulations):
    sample = np.random.normal(0, true_sigma, n)
    stds_biased.append(sample.std(ddof=0))
    stds_unbiased.append(sample.std(ddof=1))

stds_biased = np.array(stds_biased)
stds_unbiased = np.array(stds_unbiased)

# 2. Calculate expectations
print(f"\nTrue σ = {true_sigma}")
print(f"Sample size n = {n}")

print(f"\n📊 Biased Estimator (ddof=0):")
print(f"  E[ŝ] = {stds_biased.mean():.6f}")
print(f"  Bias = E[ŝ] - σ = {stds_biased.mean() - true_sigma:.6f}")

print(f"\n📊 'Unbiased' Estimator (ddof=1):")
print(f"  E[ŝ] = {stds_unbiased.mean():.6f}")
print(f"  Bias = E[ŝ] - σ = {stds_unbiased.mean() - true_sigma:.6f}")

# 3. Comparison
print("\n" + "-"*55)
print("ANALYSIS:")
print(f"  ddof=1 has LESS bias ({abs(stds_unbiased.mean() - true_sigma):.4f}")
print(f"  vs ddof=0 bias {abs(stds_biased.mean() - true_sigma):.4f})")

# 4. Why isn't ddof=1 unbiased for std?
print("\n❓ Why isn't ddof=1 perfectly unbiased?")
print("  - ddof=1 makes VARIANCE estimator unbiased: E[s²] = σ²")
print("  - But E[√s²] ≠ √E[s²] (Jensen's inequality!)")
print("  - √ is concave, so E[√X] < √E[X]")
print("  - Standard deviation estimator is always biased low!")

# Verify variance is unbiased
vars_unbiased = stds_unbiased**2
print(f"\n✅ Verification: E[s²] = {vars_unbiased.mean():.4f} (True σ² = {true_sigma**2})")

---
## Exercise 2: MLE for Poisson Distribution

**Problem**: Derive and verify MLE for Poisson(λ).

Tasks:
1. Generate 50 samples from Poisson(λ=7)
2. Write the negative log-likelihood function
3. Find MLE numerically using scipy.optimize.minimize
4. Verify it equals the sample mean

In [ ]:
# Generate Poisson data
np.random.seed(42)
true_lambda = 7
data = np.random.poisson(true_lambda, 50)

# Your solution here

### Solution

In [ ]:
# Generate Poisson data
np.random.seed(42)
true_lambda = 7
data = np.random.poisson(true_lambda, 50)

print("MLE FOR POISSON DISTRIBUTION")
print("="*55)

print(f"\nTrue λ = {true_lambda}")
print(f"Data: n={len(data)}, sum={data.sum()}, mean={data.mean():.4f}")

# 1. Define negative log-likelihood
# P(X=k|λ) = e^(-λ) * λ^k / k!
# log P = -λ + k*log(λ) - log(k!)
# Sum: -n*λ + (Σk)*log(λ) - Σlog(k!)

def neg_log_likelihood(lam, data):
    if lam <= 0:
        return np.inf
    n = len(data)
    return n * lam - data.sum() * np.log(lam)  # Ignore constant term

print("\n📝 Log-likelihood derivation:")
print("  ℓ(λ) = -nλ + (Σxᵢ)log(λ) - Σlog(xᵢ!)")
print("  dℓ/dλ = -n + (Σxᵢ)/λ = 0")
print("  ∴ λ̂_MLE = Σxᵢ/n = X̄")

# 2. Numerical optimization
from scipy.optimize import minimize_scalar

result = minimize_scalar(neg_log_likelihood, args=(data,), bounds=(0.1, 20), method='bounded')
mle_numerical = result.x

# 3. Analytical solution
mle_analytical = data.mean()

print(f"\n📊 Results:")
print(f"  Numerical MLE: λ̂ = {mle_numerical:.6f}")
print(f"  Analytical MLE: λ̂ = X̄ = {mle_analytical:.6f}")
print(f"  Match: {np.isclose(mle_numerical, mle_analytical)}")

# Visualization
lambdas = np.linspace(4, 10, 100)
nlls = [neg_log_likelihood(l, data) for l in lambdas]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lambdas, nlls, 'b-', linewidth=2)
ax.axvline(mle_analytical, color='r', linestyle='--', label=f'MLE λ̂={mle_analytical:.3f}')
ax.axvline(true_lambda, color='g', linestyle=':', label=f'True λ={true_lambda}')
ax.set_xlabel('λ')
ax.set_ylabel('Negative Log-Likelihood')
ax.set_title('MLE for Poisson: Finding the Minimum')
ax.legend()
plt.show()

---
## Exercise 3: Fisher Information

**Problem**: Calculate Fisher Information for Bernoulli(p).

Tasks:
1. Derive the Fisher Information I(p) for Bernoulli
2. Calculate the Cramér-Rao Lower Bound for p̂
3. Verify that MLE (sample proportion) achieves the bound

In [ ]:
# Parameters
true_p = 0.3
n = 100
n_simulations = 10000

# Your solution here

### Solution

In [ ]:
# Parameters
true_p = 0.3
n = 100
n_simulations = 10000

print("FISHER INFORMATION FOR BERNOULLI")
print("="*55)

# 1. Derivation
print("\n📝 Derivation:")
print("  P(X=x|p) = p^x * (1-p)^(1-x)")
print("  log P = x*log(p) + (1-x)*log(1-p)")
print("  ∂log P/∂p = x/p - (1-x)/(1-p)")
print("  ∂²log P/∂p² = -x/p² - (1-x)/(1-p)²")
print("  ")
print("  Fisher Info I(p) = -E[∂²log P/∂p²]")
print("                   = E[X]/p² + E[1-X]/(1-p)²")
print("                   = p/p² + (1-p)/(1-p)²")
print("                   = 1/p + 1/(1-p)")
print("                   = 1/(p(1-p))")

# 2. Calculate I(p) and CRLB
fisher_info_single = 1 / (true_p * (1 - true_p))
fisher_info_n = n * fisher_info_single  # For n observations
crlb = 1 / fisher_info_n  # Variance lower bound

print(f"\n📊 For p = {true_p}, n = {n}:")
print(f"  I(p) for single obs: {fisher_info_single:.4f}")
print(f"  I(p) for n obs: n × I(p) = {fisher_info_n:.4f}")
print(f"  CRLB: Var(p̂) ≥ 1/I = {crlb:.6f}")
print(f"  CRLB SE: √(CRLB) = {np.sqrt(crlb):.6f}")

# 3. Verify with simulation
np.random.seed(42)
p_estimates = []
for _ in range(n_simulations):
    sample = np.random.binomial(1, true_p, n)
    p_estimates.append(sample.mean())

p_estimates = np.array(p_estimates)
empirical_var = p_estimates.var()

print(f"\n📊 Simulation ({n_simulations} trials):")
print(f"  E[p̂] = {p_estimates.mean():.6f} (True p = {true_p})")
print(f"  Var(p̂) = {empirical_var:.6f}")
print(f"  CRLB = {crlb:.6f}")
print(f"  Ratio: Var/CRLB = {empirical_var/crlb:.4f}")

print("\n✅ MLE achieves the CRLB (ratio ≈ 1)")
print("   Sample proportion is an EFFICIENT estimator!")

---
## Exercise 4: Method of Moments for Beta Distribution

**Problem**: Estimate Beta(α, β) parameters using Method of Moments.

For Beta(α, β):
- E[X] = α/(α+β)
- Var[X] = αβ/((α+β)²(α+β+1))

Tasks:
1. Generate 200 samples from Beta(2, 5)
2. Calculate sample mean and variance
3. Derive and apply MoM estimators
4. Compare with true parameters

In [ ]:
# Generate Beta data
np.random.seed(42)
true_alpha, true_beta = 2, 5
data = np.random.beta(true_alpha, true_beta, 200)

# Your solution here

### Solution

In [ ]:
# Generate Beta data
np.random.seed(42)
true_alpha, true_beta = 2, 5
data = np.random.beta(true_alpha, true_beta, 200)

print("METHOD OF MOMENTS: BETA DISTRIBUTION")
print("="*55)

print(f"\nTrue parameters: α={true_alpha}, β={true_beta}")

# 1. Sample moments
m = data.mean()      # Sample mean
v = data.var(ddof=1)  # Sample variance

print(f"\n📊 Sample moments:")
print(f"  m = E[X] = {m:.6f}")
print(f"  v = Var[X] = {v:.6f}")

# 2. Derive MoM estimators
# Let μ = α/(α+β) and σ² = αβ/((α+β)²(α+β+1))
# From these: α = μ * (μ(1-μ)/σ² - 1)
#             β = (1-μ) * (μ(1-μ)/σ² - 1)

print("\n📝 MoM Derivation:")
print("  Let c = μ(1-μ)/σ² - 1")
print("  α̂ = μ × c")
print("  β̂ = (1-μ) × c")

c = m * (1 - m) / v - 1
alpha_mom = m * c
beta_mom = (1 - m) * c

print(f"\n📊 MoM Estimates:")
print(f"  c = {c:.4f}")
print(f"  α̂ = {alpha_mom:.4f} (True: {true_alpha})")
print(f"  β̂ = {beta_mom:.4f} (True: {true_beta})")

# 3. Compare with MLE (using scipy)
mle_alpha, mle_beta, _, _ = stats.beta.fit(data, floc=0, fscale=1)

print(f"\n📊 MLE Estimates (scipy.stats.beta.fit):")
print(f"  α̂ = {mle_alpha:.4f}")
print(f"  β̂ = {mle_beta:.4f}")

# 4. Compare errors
print("\n" + "-"*55)
print("COMPARISON:")
mom_error = np.sqrt((alpha_mom - true_alpha)**2 + (beta_mom - true_beta)**2)
mle_error = np.sqrt((mle_alpha - true_alpha)**2 + (mle_beta - true_beta)**2)
print(f"  MoM Euclidean error: {mom_error:.4f}")
print(f"  MLE Euclidean error: {mle_error:.4f}")

# Visualization
x = np.linspace(0, 1, 100)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(data, bins=30, density=True, alpha=0.5, label='Data')
ax.plot(x, stats.beta.pdf(x, true_alpha, true_beta), 'g-', lw=2, label=f'True: Beta({true_alpha},{true_beta})')
ax.plot(x, stats.beta.pdf(x, alpha_mom, beta_mom), 'r--', lw=2, label=f'MoM: Beta({alpha_mom:.2f},{beta_mom:.2f})')
ax.plot(x, stats.beta.pdf(x, mle_alpha, mle_beta), 'b:', lw=2, label=f'MLE: Beta({mle_alpha:.2f},{mle_beta:.2f})')
ax.legend()
ax.set_title('Beta Distribution: MoM vs MLE Estimates')
plt.show()

---
## Exercise 5: MSE Comparison

**Problem**: Compare MSE of different estimators for population mean.

Estimators:
- Sample mean (unbiased)
- Shrinkage estimator: (1-λ)X̄ + λ×50 (shrink toward 50)

Tasks:
1. Simulate for true μ = 52 (close to 50)
2. Calculate MSE for both estimators with λ = 0.2
3. Find which has lower MSE
4. Explain bias-variance tradeoff

In [ ]:
# Parameters
true_mu = 52
true_sigma = 15
n = 20
shrink_lambda = 0.2
shrink_target = 50
n_simulations = 10000

# Your solution here

### Solution

In [ ]:
# Parameters
true_mu = 52
true_sigma = 15
n = 20
shrink_lambda = 0.2
shrink_target = 50
n_simulations = 10000

print("MSE COMPARISON: SAMPLE MEAN vs SHRINKAGE")
print("="*55)

print(f"\nSetup:")
print(f"  True μ = {true_mu}, σ = {true_sigma}, n = {n}")
print(f"  Shrinkage: (1-λ)X̄ + λ×{shrink_target}, λ = {shrink_lambda}")

np.random.seed(42)

sample_means = []
shrinkage_estimates = []

for _ in range(n_simulations):
    sample = np.random.normal(true_mu, true_sigma, n)
    xbar = sample.mean()
    shrinkage = (1 - shrink_lambda) * xbar + shrink_lambda * shrink_target
    sample_means.append(xbar)
    shrinkage_estimates.append(shrinkage)

sample_means = np.array(sample_means)
shrinkage_estimates = np.array(shrinkage_estimates)

# Calculate bias, variance, MSE
print("\n" + "="*55)
print(f"{'Metric':<20} {'Sample Mean':>15} {'Shrinkage':>15}")
print("-"*55)

# Bias
bias_mean = sample_means.mean() - true_mu
bias_shrink = shrinkage_estimates.mean() - true_mu
print(f"{'Bias':<20} {bias_mean:>15.6f} {bias_shrink:>15.6f}")

# Variance
var_mean = sample_means.var()
var_shrink = shrinkage_estimates.var()
print(f"{'Variance':<20} {var_mean:>15.6f} {var_shrink:>15.6f}")

# MSE = Bias² + Variance
mse_mean = ((sample_means - true_mu)**2).mean()
mse_shrink = ((shrinkage_estimates - true_mu)**2).mean()
print(f"{'MSE':<20} {mse_mean:>15.6f} {mse_shrink:>15.6f}")

print("-"*55)

# Analysis
print("\n📊 Analysis:")
print(f"  Sample Mean: Unbiased (bias≈0), higher variance")
print(f"  Shrinkage: Biased toward {shrink_target}, lower variance")

if mse_shrink < mse_mean:
    print(f"\n✅ Shrinkage wins! MSE reduction: {(mse_mean-mse_shrink)/mse_mean*100:.1f}%")
else:
    print(f"\n✅ Sample mean wins! MSE better by: {(mse_shrink-mse_mean)/mse_shrink*100:.1f}%")

print("\n💡 Bias-Variance Tradeoff:")
print("  - Adding bias reduces variance")
print("  - Total error (MSE) can be lower with biased estimator!")
print("  - This is the foundation of regularization in ML")

# Visualization
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(sample_means, bins=50, alpha=0.5, label=f'Sample Mean (MSE={mse_mean:.2f})', density=True)
ax.hist(shrinkage_estimates, bins=50, alpha=0.5, label=f'Shrinkage (MSE={mse_shrink:.2f})', density=True)
ax.axvline(true_mu, color='green', linestyle='-', linewidth=2, label=f'True μ={true_mu}')
ax.axvline(shrink_target, color='orange', linestyle=':', linewidth=2, label=f'Shrink target={shrink_target}')
ax.legend()
ax.set_title('Bias-Variance Tradeoff: Sample Mean vs Shrinkage Estimator')
ax.set_xlabel('Estimate')
plt.show()